<!--nav--> [🗺 Learning path](README.md) · **42/43** · ◀ [The Optimization Stack](./The_Optimization_Stack.ipynb) · [VLM Optimization Techniques](./VLM_Optimization_Techniques.ipynb) ▶

# Serving Vision-Language Models: The Token Explosion

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sugeerth/gpu-training-notebooks/blob/main/VLM_Serving_Token_Explosion.ipynb)

Everything in notebooks 22–41 assumed text. Vision-language models break that in ways that are
easy to miss and expensive to discover in production:

> A user uploads one photo and types **five words**. Your engine sees a prompt of **1,300 tokens**
> — or, at high resolution, **7,000**. Their "short question" is a small document.

That single fact rewrites TTFT, KV budgeting, batching, and cost. And VLMs add a whole compute
phase that text models don't have, which sits *outside* everything the roofline notebook measured.

| Part | What you'll learn |
|---|---|
| **1** | The two-tower anatomy: four phases, not two |
| **2** | **Image → token math**, for real VLM families (fixed grid, AnyRes, dynamic, tiling) |
| **3** | **Where VLM time actually goes** — including the CPU-side bottleneck nobody expects |
| **4** | The interactive resolution explorer: pixels → tokens → TTFT → KV → cost |
| **5** | Why variable image sizes wreck batching, and what schedulers do about it |
| **6** | KV budgeting for images, and the multi-image cliff |
| **7** | vLLM's multimodal knobs, and how they interact with chunked prefill |

**Runs on:** any CPU — all modeling. GPU-gated cells measure a real VLM.

**Companion:** [notebook 43](./VLM_Optimization_Techniques.ipynb) covers the optimizations.
Training VLMs is [notebook 17](./Simple_MultiGPU_Multimodal.ipynb) and
[18](./Multimodal_LoRA_QLoRA_DPO.ipynb) — this is what happens after.

In [ ]:
import math, json, random, uuid, statistics
from collections import defaultdict
from IPython.display import HTML, display

D3_URL = "https://cdn.jsdelivr.net/npm/d3@7/dist/d3.min.js"
def show_d3(js, data=None, height=420):
    div = f"viz_{uuid.uuid4().hex[:10]}"
    html = f'''
<div id="{div}" style="width:100%;max-width:920px;font-family:system-ui,sans-serif"></div>
<script>
(function() {{
  function run() {{
    const d3 = window.d3, root = d3.select("#{div}"), data = {json.dumps(data)};
    const W = (document.getElementById("{div}").clientWidth || 880), H = {height};
    try {{ {js} }} catch (e) {{ root.append("pre").style("color","crimson").text("viz error: " + e); }}
  }}
  if (window.d3) run();
  else {{ const s = document.createElement("script"); s.src = "{D3_URL}"; s.onload = run;
          s.onerror = () => document.getElementById("{div}").textContent = "Could not load D3.";
          document.head.appendChild(s); }}
}})();
</script>'''
    display(HTML(html))
print("ready")

## Part 1 · Four phases, not two

Text decode has two phases (nb 22: prefill, decode). A VLM request has **four**, and the two new
ones have completely different performance characteristics:

```
 ┌─ 1. PREPROCESS ──── decode JPEG, resize, normalize, tile        (CPU! often overlooked)
 │  2. VISION ENCODE ── ViT forward over image patches             (GPU, COMPUTE-bound, batchable)
 │  3. PROJECT ──────── MLP/resampler: vision dims → LLM dims      (GPU, trivial)
 │  4. PREFILL ─────── LLM over [image tokens + text tokens]       (GPU, compute-bound, big)
 └─ 5. DECODE ──────── the usual one-token-at-a-time loop          (GPU, memory-bound)
```

Three consequences that catch teams out:

1. **Phase 1 runs on the CPU.** Image decode + resize is 10–50 ms of CPU work per image. Your GPU
   might do the ViT in 3 ms. Under load, the *preprocessing* becomes the bottleneck — and no GPU
   metric shows it (nb 27's dashboard looks fine while requests queue).
2. **Phase 2 is compute-bound and independent per image.** It batches beautifully, and it's
   *cacheable* — the same image sent twice needs encoding once (nb 43).
3. **Phase 4 is prefill, but the sequence is dominated by image tokens.** All of notebook 35's
   long-context reasoning applies, except the length comes from pixels rather than words.

## Part 2 · Image → token math

This is the number that determines everything, and each VLM family computes it differently:

| Family | Scheme | Tokens |
|---|---|---|
| **LLaVA-1.5** | fixed 336², patch 14 | `(336/14)² = 576`, always |
| **LLaVA-NeXT** | AnyRes: base + up to N tiles | `576 × (1 + tiles)`, up to ~2,880 |
| **Qwen2/2.5-VL** | native dynamic resolution, 2×2 merge | `≈ (H/28) × (W/28)`, bounded by min/max pixels |
| **InternVL** | 448² tiles + pixel shuffle | `256 × (tiles + thumbnail)`, up to ~3,300 |

The design tension is identical across all of them: **more tokens = better fine detail = quadratically
more prefill and linearly more KV.**

In [ ]:
import math

def llava_15(w, h):
    return 576                                            # fixed grid, resolution-independent

def llava_next(w, h, max_tiles=4, base=576):
    # AnyRes: pick a grid that best matches the aspect ratio, plus a global thumbnail.
    best, best_waste = (1, 1), float("inf")
    for gw in range(1, max_tiles + 1):
        for gh in range(1, max_tiles + 1):
            if gw * gh > max_tiles:
                continue
            scale = min(gw * 336 / w, gh * 336 / h)
            used = (w * scale) * (h * scale)
            waste = (gw * 336) * (gh * 336) - used
            if waste < best_waste:
                best, best_waste = (gw, gh), waste
    return base * (1 + best[0] * best[1])                 # thumbnail + tiles

def qwen2_vl(w, h, min_pixels=256 * 28 * 28, max_pixels=1280 * 28 * 28):
    # Native dynamic resolution: resize so pixel count sits in [min, max], patch 14 with 2x2 merge.
    pixels = w * h
    scale = 1.0
    if pixels > max_pixels:
        scale = math.sqrt(max_pixels / pixels)
    elif pixels < min_pixels:
        scale = math.sqrt(min_pixels / pixels)
    ww, hh = max(28, int(w * scale)), max(28, int(h * scale))
    return max(1, (ww // 28) * (hh // 28))

def internvl(w, h, max_tiles=12, min_tiles=1, tile=448, tokens_per_tile=256):
    # InternVL's dynamic tiling: choose the tile grid whose ASPECT RATIO best matches the
    # image, and only use a large grid if the image actually has the pixels to justify it.
    # (Matching on wasted area alone upscales a thumbnail into a 12-tile grid - which is
    #  backwards, and was a real bug in the first draft of this notebook.)
    ar = w / h
    candidates = sorted({(i, j) for n in range(min_tiles, max_tiles + 1)
                         for i in range(1, n + 1) for j in range(1, n + 1)
                         if min_tiles <= i * j <= max_tiles},
                        key=lambda g: g[0] * g[1])
    best, best_diff = (1, 1), float("inf")
    area = w * h
    for gw, gh in candidates:
        diff = abs(ar - gw / gh)
        if diff < best_diff:
            best, best_diff = (gw, gh), diff
        elif diff == best_diff:
            # tie-break toward more tiles ONLY if the image has enough pixels to fill them
            if area > 0.5 * tile * tile * gw * gh:
                best = (gw, gh)
    n_tiles = best[0] * best[1]
    thumbnail = 1 if n_tiles > 1 else 0                   # global thumbnail only when tiled
    return tokens_per_tile * (n_tiles + thumbnail)

FAMILIES = {"LLaVA-1.5": llava_15, "LLaVA-NeXT": llava_next,
            "Qwen2-VL": qwen2_vl, "InternVL": internvl}

IMAGES = [("thumbnail",  320,  240), ("phone photo", 1080, 1440),
          ("screenshot", 1920, 1080), ("scan / document", 2480, 3508),
          ("4K photo",   3840, 2160)]

print(f"{'image':<18}{'pixels':>10}" + "".join(f"{f:>13}" for f in FAMILIES))
print("-" * 76)
for name, w, h in IMAGES:
    row = [fn(w, h) for fn in FAMILIES.values()]
    print(f"{name:<18}{w*h/1e6:>8.1f}MP" + "".join(f"{v:>13,}" for v in row))

print("\nA 5-word question about an A4 scan is a 3,000+ token prompt on most modern VLMs.")
print("For comparison, this entire paragraph is about 40 tokens.")
print("\nNote LLaVA-1.5's column: constant regardless of input. That is cheap and predictable,")
print("and it is also why it cannot read small text in a document - the tension in one table.")

## Part 3 · Where the time actually goes

Now put costs on the phases. The formulas come from notebooks 22 and 39; the new terms are the ViT
and — crucially — the CPU preprocessing that no GPU dashboard will show you.

In [ ]:
HW = {   # same constants as nb 32/39/40 (cross-checked by tools/audit_consistency.py)
  "T4":        dict(bw=0.32e12, tf=65e12),
  "A100 80GB": dict(bw=2.04e12, tf=312e12),
  "H100 SXM":  dict(bw=3.35e12, tf=990e12),
  "MI300X":    dict(bw=5.30e12, tf=1307e12),
}
VLM = {
  # vision encoder params, LLM params, layers, kv_heads, head_dim
  "LLaVA-1.5-7B":   dict(vit=0.30e9, llm=7e9,  layers=32, kv_heads=32, head_dim=128, tokens=llava_15),
  "Qwen2-VL-7B":    dict(vit=0.68e9, llm=7e9,  layers=28, kv_heads=4,  head_dim=128, tokens=qwen2_vl),
  "InternVL2-8B":   dict(vit=0.30e9, llm=7e9,  layers=32, kv_heads=8,  head_dim=128, tokens=internvl),
}
FLOP_EFF, BW_EFF = 0.60, 0.75

def preprocess_ms(w, h):
    # JPEG decode + resize + normalize on one CPU core. Roughly a fixed cost plus a
    # per-megapixel cost - a 4K photo is genuinely expensive to decode, a thumbnail is not.
    return 6.0 + 6.5 * (w * h / 1e6)

CPU_PREPROCESS_MS = preprocess_ms(1920, 1080)

def vlm_request(model, gpu, w, h, text_tokens=16, out_tokens=64, n_images=1,
                cpu_workers=4, preprocess_ms=CPU_PREPROCESS_MS):
    m, g = VLM[model], HW[gpu]
    flops = g["tf"] * FLOP_EFF
    bw = g["bw"] * BW_EFF
    img_tokens = m["tokens"](w, h) * n_images

    # 1. CPU preprocessing - parallel across workers, but it is NOT free and NOT on the GPU
    preprocess_s = (preprocess_ms / 1000) * math.ceil(n_images / max(1, cpu_workers))
    # 2. vision encoder: a ViT forward over the patches
    vit_s = (2 * m["vit"] * img_tokens) / flops
    # 4. LLM prefill over image + text tokens
    prompt_tokens = img_tokens + text_tokens
    prefill_s = (2 * m["llm"] * prompt_tokens) / flops
    # 5. decode
    decode_step_s = (m["llm"] * 2) / bw
    decode_s = decode_step_s * out_tokens
    kv_per_token = 2 * m["layers"] * m["kv_heads"] * m["head_dim"] * 2

    return {"img_tokens": img_tokens, "prompt_tokens": prompt_tokens,
            "preprocess_s": preprocess_s, "vit_s": vit_s, "prefill_s": prefill_s,
            "decode_s": decode_s, "ttft_s": preprocess_s + vit_s + prefill_s,
            "total_s": preprocess_s + vit_s + prefill_s + decode_s,
            "kv_mb": kv_per_token * prompt_tokens / 1e6}

print("One 1920x1080 screenshot + a 16-token question, 64 tokens out, on an A100:\n")
print(f"{'model':<16}{'img tok':>9}{'preproc':>10}{'ViT':>9}{'prefill':>10}{'decode':>10}"
      f"{'TTFT':>9}{'KV':>9}")
print("-" * 84)
for name in VLM:
    r = vlm_request(name, "A100 80GB", 1920, 1080)
    print(f"{name:<16}{r['img_tokens']:>9,}{r['preprocess_s']*1000:>9.0f}ms{r['vit_s']*1000:>8.0f}ms"
          f"{r['prefill_s']*1000:>9.0f}ms{r['decode_s']*1000:>9.0f}ms{r['ttft_s']*1000:>8.0f}ms"
          f"{r['kv_mb']:>8.0f}MB")

r = vlm_request("Qwen2-VL-7B", "A100 80GB", 1920, 1080)
print(f"\nLook at the preprocessing column. On Qwen2-VL it is "
      f"{r['preprocess_s']/r['ttft_s']:.0%} of TTFT - and it runs on the CPU,")
print("so every GPU metric in notebook 27's dashboard says the GPU is idle while users wait.")
print("\nThis is the single most common VLM serving surprise: you scale GPUs and nothing improves,")
print("because the queue is in your image preprocessing pool.")

### When does the CPU actually bind?

It is tempting to declare "CPU preprocessing is the hidden bottleneck" and move on. The honest
answer is **it depends**, and the dependency is worth understanding precisely — because the regimes
where it binds are exactly the ones people deploy into without noticing.

The comparison has to be like-for-like. Decode is amortized across concurrently-running requests
(nb 22's continuous batching), so the fair comparison is:

- **CPU stage**: `preprocess_ms / workers` per image — pure per-request serial work
- **GPU prefill stage**: ViT + LLM prefill — also per-request serial work

Whichever stage is slower sets the pipeline's throughput.

In [ ]:
def binding_stage(model, gpu, w, h, cpu_workers):
    m, g = VLM[model], HW[gpu]
    toks = m["tokens"](w, h)
    cpu_ms = preprocess_ms(w, h) / cpu_workers
    gpu_ms = ((2 * m["vit"] * toks + 2 * m["llm"] * (toks + 16))
              / (g["tf"] * FLOP_EFF)) * 1000
    return {"tokens": toks, "cpu_ms": cpu_ms, "gpu_ms": gpu_ms,
            "rps": 1000 / max(cpu_ms, gpu_ms),
            "bound_by": "CPU preprocess" if cpu_ms > gpu_ms else "GPU"}

print("Which stage binds? (A100, 4 preprocessing workers)\n")
print(f"{'model':<16}{'image':<18}{'tokens':>8}{'CPU ms':>9}{'GPU ms':>9}{'req/s':>8}   bound by")
print("-" * 82)
for model in ("Qwen2-VL-7B", "LLaVA-1.5-7B"):
    for label, w, h in [("thumbnail 320x240", 320, 240), ("phone 1080x1440", 1080, 1440),
                        ("4K photo 3840x2160", 3840, 2160)]:
        b = binding_stage(model, "A100 80GB", w, h, cpu_workers=4)
        print(f"{model:<16}{label:<18}{b['tokens']:>8,}{b['cpu_ms']:>9.1f}{b['gpu_ms']:>9.1f}"
              f"{b['rps']:>8.1f}   {b['bound_by']}")
    print()

print("The pattern, stated honestly:")
print("  - LLaVA-1.5 (fixed 576 tokens) does LESS GPU work as images grow, while the CPU cost")
print("    grows with megapixels -> it flips to CPU-bound on large uploads.")
print("  - Qwen2-VL scales its tokens with the image, so the GPU stage grows too and usually")
print("    stays the binding stage - until you cap max_pixels (Part 7), which pushes it back")
print("    toward CPU-bound.")
print("\nSo the rule is not 'the CPU is always the bottleneck'. It is:")
print("  → capping resolution moves work from the GPU to the (fixed) CPU cost,")
print("    and a fixed-grid model is CPU-bound on big images by construction.")
print("  Measure both stages before you scale either one.")

print("\nWorker count sweep, LLaVA-1.5 on 4K uploads (the CPU-bound regime):\n")
print(f"{'workers':>9}{'CPU ms':>9}{'GPU ms':>9}{'req/s':>8}   bound by")
print("-" * 52)
for wk in (1, 2, 4, 8, 16):
    b = binding_stage("LLaVA-1.5-7B", "A100 80GB", 3840, 2160, cpu_workers=wk)
    print(f"{wk:>9}{b['cpu_ms']:>9.1f}{b['gpu_ms']:>9.1f}{b['rps']:>8.1f}   {b['bound_by']}")
print("\nAdding workers helps until the GPU stage takes over, then stops. Classic pipelining -")
print("and the reason 'we added GPUs and throughput didn't move' happens in VLM deployments.")
print("\nMitigations that help in EITHER regime:")
print("  - accept client-side pre-resized images (removes megapixels before they reach you)")
print("  - cache decoded/encoded images (nb 43) - a repeat image skips BOTH stages")
print("  - watch queue depth next to GPU utilisation (nb 27), or a CPU-bound stall is invisible")

## Part 4 · The resolution explorer

Resolution is the master dial of VLM serving. Drag it and watch tokens, TTFT, KV and cost move
together — this is the trade every VLM product makes, usually without measuring it.

In [ ]:
grid = []
for family, fn in FAMILIES.items():
    for side in (256, 384, 512, 768, 1024, 1536, 2048, 3072, 4096):
        w, h = side, int(side * 0.75)
        toks = fn(w, h)
        model = {"LLaVA-1.5": "LLaVA-1.5-7B", "LLaVA-NeXT": "LLaVA-1.5-7B",
                 "Qwen2-VL": "Qwen2-VL-7B", "InternVL": "InternVL2-8B"}[family]
        m = VLM[model]
        vit_s = (2 * m["vit"] * toks) / (HW["A100 80GB"]["tf"] * FLOP_EFF)
        prefill_s = (2 * m["llm"] * (toks + 16)) / (HW["A100 80GB"]["tf"] * FLOP_EFF)
        kv_mb = 2 * m["layers"] * m["kv_heads"] * m["head_dim"] * 2 * (toks + 16) / 1e6
        grid.append({"family": family, "side": side, "tokens": toks,
                     "vit_ms": round(vit_s * 1000, 2), "prefill_ms": round(prefill_s * 1000, 2),
                     "preprocess_ms": CPU_PREPROCESS_MS,
                     "ttft_ms": round((vit_s + prefill_s) * 1000 + CPU_PREPROCESS_MS, 1),
                     "kv_mb": round(kv_mb, 1)})
print(f"precomputed {len(grid)} (family x resolution) points")

JS = r'''
const fams = [...new Set(data.map(d=>d.family))];
const ctr = root.append("div").style("font","13px system-ui").style("margin-bottom","8px");
ctr.append("span").text("family: ");
const sel = ctr.append("select").style("font-size","13px");
sel.selectAll("o").data(fams).join("option").attr("value",d=>d).text(d=>d);
sel.property("value","Qwen2-VL");
const M = {top: 20, right: 66, bottom: 46, left: 62};
const iw = W - M.left - M.right, ih = H - M.top - M.bottom - 26;
const svg = root.append("svg").attr("width",W).attr("height",H).append("g")
    .attr("transform",`translate(${M.left},${M.top})`);
const x = d3.scaleBand().range([0,iw]).padding(0.2);
const y = d3.scaleLinear().range([ih,0]);
const y2 = d3.scaleLinear().range([ih,0]);
const xAxis = svg.append("g").attr("transform",`translate(0,${ih})`);
const yAxis = svg.append("g");
const yAxis2 = svg.append("g").attr("transform",`translate(${iw},0)`);
svg.append("text").attr("x",iw/2).attr("y",ih+38).attr("text-anchor","middle")
   .style("font-size","12px").text("image width (px, 4:3 aspect)");
svg.append("text").attr("transform","rotate(-90)").attr("x",-ih/2).attr("y",-46)
   .attr("text-anchor","middle").style("font-size","12px").text("TTFT (ms, stacked)");
svg.append("text").attr("transform","rotate(90)").attr("x",ih/2).attr("y",-iw-46)
   .attr("text-anchor","middle").style("font-size","12px").style("fill","#6a1b9a")
   .text("image tokens");
const readout = root.append("div").style("font","12px ui-monospace,monospace").style("color","#455a64");
const COLORS = {preprocess_ms:"#ff9800", vit_ms:"#1976d2", prefill_ms:"#c62828"};
const KEYS = ["preprocess_ms","vit_ms","prefill_ms"];
KEYS.forEach((k,i)=>{
  svg.append("rect").attr("x",6+i*118).attr("y",-14).attr("width",10).attr("height",10)
     .attr("fill",COLORS[k]).attr("rx",2);
  svg.append("text").attr("x",20+i*118).attr("y",-5).style("font-size","10.5px")
     .text({preprocess_ms:"CPU preprocess",vit_ms:"vision encoder",prefill_ms:"LLM prefill"}[k]);
});
const tokLine = svg.append("path").attr("fill","none").attr("stroke","#6a1b9a")
    .attr("stroke-width",2).attr("stroke-dasharray","4 3");
function draw() {
  const fam = sel.property("value");
  const rows = data.filter(d=>d.family===fam).sort((a,b)=>a.side-b.side);
  x.domain(rows.map(d=>d.side));
  y.domain([0, d3.max(rows,d=>d.ttft_ms)*1.15]);
  y2.domain([0, d3.max(rows,d=>d.tokens)*1.15]);
  xAxis.call(d3.axisBottom(x));
  yAxis.call(d3.axisLeft(y).ticks(5));
  yAxis2.call(d3.axisRight(y2).ticks(5,"~s"));
  const stack = d3.stack().keys(KEYS);
  svg.selectAll(".layer").remove();
  svg.selectAll("layer").data(stack(rows)).join("g").attr("class","layer")
     .attr("fill",d=>COLORS[d.key])
     .selectAll("rect").data(d=>d.map(v=>({...v,key:d.key}))).join("rect")
     .attr("x",d=>x(d.data.side)).attr("width",x.bandwidth())
     .attr("y",d=>y(d[1])).attr("height",d=>Math.max(0,y(d[0])-y(d[1])))
     .append("title").text(d=>`${d.key}: ${(d[1]-d[0]).toFixed(1)} ms`);
  tokLine.datum(rows).attr("d", d3.line().x(d=>x(d.side)+x.bandwidth()/2).y(d=>y2(d.tokens)));
  const lo = rows[0], hi = rows[rows.length-1];
  readout.text(
`${lo.side}px : ${d3.format(",")(lo.tokens)} tokens · TTFT ${lo.ttft_ms.toFixed(0)} ms · KV ${lo.kv_mb} MB
${hi.side}px : ${d3.format(",")(hi.tokens)} tokens · TTFT ${hi.ttft_ms.toFixed(0)} ms · KV ${hi.kv_mb} MB
=> ${(hi.tokens/Math.max(lo.tokens,1)).toFixed(1)}x tokens, ${(hi.ttft_ms/lo.ttft_ms).toFixed(1)}x TTFT, ${(hi.kv_mb/Math.max(lo.kv_mb,0.01)).toFixed(1)}x KV`);
}
sel.on("change", draw); draw();
'''
show_d3(JS, grid, height=400)

**Switch between families and watch the shapes differ.**

- **LLaVA-1.5** is a flat line: 576 tokens at any resolution. Predictable, cheap, and blind to fine
  detail — a document scan gets the same 576 tokens as a thumbnail.
- **Qwen2-VL / InternVL** scale with pixels, so TTFT and KV grow with what the user uploads. Your
  cost per request is now **controlled by your users' camera**, unless you bound it (Part 7).
- At small resolutions, the **orange CPU preprocessing block dominates** — a fixed cost that doesn't
  shrink with the image. Below ~512px you are mostly paying to decode a JPEG.

## Part 5 · Variable length wrecks batching

Text prompts vary; image prompts vary *wildly*. One request is 576 tokens, the next is 7,000. This
is notebook 22's straggler problem with the variance turned up:

In [ ]:
import random, statistics
random.seed(5)

def simulate_batch(policy, n_requests=240, batch_size=16, steps=4000):
    # Mixed traffic: thumbnails, phone photos, screenshots and document scans.
    mix = [(320, 240, 0.30), (1080, 1440, 0.35), (1920, 1080, 0.25), (2480, 3508, 0.10)]
    reqs = []
    for i in range(n_requests):
        r = random.random(); acc = 0
        for w, h, p in mix:
            acc += p
            if r <= acc:
                reqs.append(qwen2_vl(w, h)); break
        else:
            reqs.append(qwen2_vl(1080, 1440))

    queue, t, done, wasted = list(reqs), 0, 0, 0
    while queue and t < steps:
        if policy == "naive_fixed":
            # pad every request in the batch to the largest in it (the classic mistake)
            batch = queue[:batch_size]; del queue[:batch_size]
            longest = max(batch)
            cost = longest * len(batch)
            wasted += cost - sum(batch)
        elif policy == "length_bucketed":
            # sort so similar sizes travel together - far less padding
            queue.sort()
            batch = queue[:batch_size]; del queue[:batch_size]
            longest = max(batch)
            cost = longest * len(batch)
            wasted += cost - sum(batch)
        elif policy == "token_budget":
            # continuous batching by TOKEN budget, not request count (what vLLM does)
            budget, batch = 16384, []
            while queue and sum(batch) + queue[0] <= budget:
                batch.append(queue.pop(0))
            if not batch:
                batch = [queue.pop(0)]
            cost = sum(batch)
        t += math.ceil(cost / 8192)
        done += len(batch)
    return {"policy": policy, "steps": t, "done": done,
            "wasted_tokens": wasted, "waste_frac": wasted / max(1, wasted + sum(reqs))}

print("240 mixed image requests (thumbnails to A4 scans), Qwen2-VL token counts:\n")
print(f"{'batching policy':<20}{'steps':>8}{'padding waste':>16}")
print("-" * 46)
for pol in ("naive_fixed", "length_bucketed", "token_budget"):
    s = simulate_batch(pol)
    print(f"{pol:<20}{s['steps']:>8}{s['waste_frac']:>15.0%}")

print("\nPadding every request to the batch maximum wastes a large share of your prefill compute")
print("when image sizes are this variable - and image sizes ARE this variable, because users")
print("upload whatever their phone produced.")
print("\nvLLM schedules by TOKEN BUDGET (--max-num-batched-tokens), which is why the third row")
print("has no padding waste at all: batches are formed to fill a token budget, not a request count.")
print("If you are writing your own batching layer for a VLM, this is the design to copy (nb 22).")

## Part 6 · KV budgeting for images

Image tokens are ordinary tokens once they reach the LLM — they occupy KV exactly like words
(nb 22's formula, unchanged). The difference is **how many arrive at once**.

In [ ]:
def kv_capacity(model, gpu_vram_gb, w, h, n_images=1, text_tokens=64, out_tokens=256,
                weight_bytes=2, util=0.9):
    m = VLM[model]
    weights_gb = (m["llm"] + m["vit"]) * weight_bytes / 1e9
    pool_gb = gpu_vram_gb * util - weights_gb - 1.5
    kv_per_token = 2 * m["layers"] * m["kv_heads"] * m["head_dim"] * 2
    seq_tokens = m["tokens"](w, h) * n_images + text_tokens + out_tokens
    per_req_mb = kv_per_token * seq_tokens / 1e6
    return {"weights_gb": round(weights_gb, 1), "pool_gb": round(pool_gb, 1),
            "seq_tokens": seq_tokens, "per_req_mb": round(per_req_mb, 1),
            "concurrent": int(pool_gb * 1000 // per_req_mb) if per_req_mb else 0}

print("Concurrent requests on an 80GB A100 (weights + KV), by images per request:\n")
print(f"{'model':<16}{'images':>8}{'seq tokens':>12}{'KV/req':>10}{'concurrent':>12}")
print("-" * 60)
for name in VLM:
    for n in (1, 2, 4, 8):
        c = kv_capacity(name, 80, 1920, 1080, n_images=n)
        print(f"{name if n == 1 else '':<16}{n:>8}{c['seq_tokens']:>12,}{c['per_req_mb']:>9.0f}MB"
              f"{c['concurrent']:>12}")
    print()

print("Two things to take away:")
print("  1. LLaVA-1.5's MHA attention (32 KV heads) costs ~8x the KV of Qwen2-VL's GQA (4 heads)")
print("     at the same token count - the architecture choice from nb 22, now with pictures.")
print("  2. Multi-image and video requests fall off a cliff. An 8-image request can consume more")
print("     KV than dozens of text conversations, so ONE user can evict everyone else (nb 27).")
print("     This is why --limit-mm-per-prompt exists, and why you should set it deliberately.")

## Part 7 · vLLM's multimodal knobs

```bash
vllm serve Qwen/Qwen2-VL-7B-Instruct \
  --limit-mm-per-prompt '{"image": 4}' \          # hard cap: one request can't eat the pool
  --max-model-len 8192 \                          # must cover image tokens + text + output
  --mm-processor-kwargs '{"max_pixels": 1003520}' \ # bound resolution server-side (see below)
  --gpu-memory-utilization 0.90
```

| Knob | What it protects you from | Interacts with |
|---|---|---|
| `--limit-mm-per-prompt` | one request consuming the whole KV pool | KV budgeting (Part 6) |
| `max_pixels` / `min_pixels` | users' 4K uploads setting your cost | resolution explorer (Part 4) |
| `--max-model-len` | requests that can't fit at all | image token math (Part 2) |
| `--max-num-batched-tokens` | prefill spikes freezing decode | chunked prefill (nb 35) |
| multimodal processor cache | re-encoding identical images | nb 43 |

**The interaction people miss:** chunked prefill (nb 35) and image tokens. A single large image
produces thousands of prefill tokens *that cannot be split across requests*, so a big image lands as
one dense prefill burst. If your streaming users stutter whenever someone uploads a document scan,
that's what you're seeing — lower `--max-num-batched-tokens` and accept slower TTFT for the uploader.

**`max_pixels` is the highest-leverage flag in VLM serving**, and it's the one most people never
set. It converts "cost determined by our users' cameras" into "cost we chose":

In [ ]:
print("Server-side max_pixels caps, applied to a 4K upload (3840x2160) on Qwen2-VL:\n")
print(f"{'max_pixels setting':<28}{'tokens':>9}{'TTFT':>10}{'KV/req':>10}   note")
print("-" * 78)
for label, mp in [("uncapped (max 12.8M px)", 16384 * 28 * 28),
                  ("1280*28*28 (default)", 1280 * 28 * 28),
                  ("768*28*28", 768 * 28 * 28),
                  ("512*28*28", 512 * 28 * 28),
                  ("256*28*28 (aggressive)", 256 * 28 * 28)]:
    toks = qwen2_vl(3840, 2160, max_pixels=mp)
    m = VLM["Qwen2-VL-7B"]
    ttft = (2 * m["vit"] * toks + 2 * m["llm"] * (toks + 16)) / (HW["A100 80GB"]["tf"] * FLOP_EFF)
    kv = 2 * m["layers"] * m["kv_heads"] * m["head_dim"] * 2 * toks / 1e6
    note = ("detail preserved" if toks > 1200 else
            "fine text may be lost" if toks > 500 else "thumbnail-grade")
    print(f"{label:<28}{toks:>9,}{ttft*1000+CPU_PREPROCESS_MS:>9.0f}ms{kv:>9.0f}MB   {note}")

print("\nThe span from uncapped to aggressive is over an order of magnitude in cost per request.")
print("Pick the cap from your TASK, not from the default:")
print("  - document / OCR / chart reading  -> high cap, you need the pixels")
print("  - 'what is in this photo?'        -> low cap works fine and costs a fraction")
print("  - mixed traffic                   -> route by task (nb 43 Part 4)")

## Part 8 · Measure a real VLM

The GPU-gated cell below serves a small VLM and measures the real phase split, so you can compare
against the model above.

In [ ]:
# GPU-ONLY: measure image token counts and TTFT for a real VLM.
# GPU-gated: degrade gracefully when PyTorch is absent, not just when the GPU is.
try:
    import torch
    HAS_GPU = torch.cuda.is_available()
except ImportError:
    torch = None
    HAS_GPU = False
    print("PyTorch is not installed here - skipping the GPU section.")
if not HAS_GPU:
    print("No GPU - Parts 1-7 gave you the full model.")
    print("On a T4/A100 this cell serves Qwen2-VL-2B and measures the real phase split.")
else:
    import time, io as _io, urllib.request
    from PIL import Image
    try:
        from transformers import AutoProcessor
        MODEL = "Qwen/Qwen2-VL-2B-Instruct"
        proc = AutoProcessor.from_pretrained(MODEL)

        # A synthetic image at several resolutions - no network dependency on a dataset.
        def make_image(w, h):
            import numpy as np
            arr = (np.random.default_rng(0).random((h, w, 3)) * 255).astype("uint8")
            return Image.fromarray(arr)

        print(f"{'resolution':<14}{'image tokens':>14}{'CPU preprocess':>17}")
        print("-" * 46)
        for side in (256, 512, 1024, 1536):
            img = make_image(side, int(side * 0.75))
            t0 = time.perf_counter()
            inputs = proc(text=["<|vision_start|><|image_pad|><|vision_end|>Describe."],
                          images=[img], return_tensors="pt")
            dt = time.perf_counter() - t0
            n_img_tokens = int((inputs["input_ids"] == proc.tokenizer.convert_tokens_to_ids(
                "<|image_pad|>")).sum()) if "input_ids" in inputs else -1
            predicted = qwen2_vl(side, int(side * 0.75))
            print(f"{side}x{int(side*0.75):<9}{n_img_tokens:>10,} (model said {predicted:,}){dt*1000:>10.0f}ms")

        print("\nThe 'CPU preprocess' column is the phase-1 cost from Part 3, measured on YOUR box.")
        print("Compare it to the GPU columns: if it is comparable or larger, your preprocessing")
        print("pool - not your GPU - sets your throughput ceiling.")
    except Exception as exc:
        print(f"Could not load the processor ({type(exc).__name__}: {exc}).")
        print("The modeling in Parts 1-7 stands on its own; this cell is a bonus measurement.")

## Recap

1. **A VLM request has four phases**, and two of them (CPU preprocessing, vision encoding) are
   invisible to every text-serving mental model.
2. **CPU image preprocessing is a real and frequently-binding bottleneck.** You may need ~8×
   more preprocessing workers than you'd guess before the GPU is even the limit.
3. **Image → token counts vary by an order of magnitude** across families and resolutions. A 5-word
   question about a scan is a 3,000-token prompt.
4. **Variable image sizes make padding-based batching very wasteful.** Schedule by token budget.
5. **Multi-image requests fall off a KV cliff** — cap them with `--limit-mm-per-prompt`.
6. **`max_pixels` is the highest-leverage flag** in VLM serving, and it should come from your task.

### Further reading
- [LLaVA-1.5](https://arxiv.org/abs/2310.03744) · [LLaVA-NeXT (AnyRes)](https://llava-vl.github.io/blog/2024-01-30-llava-next/)
- [Qwen2-VL](https://arxiv.org/abs/2409.12191) (naive dynamic resolution) · [InternVL](https://arxiv.org/abs/2312.14238) (tiling)
- [vLLM multimodal inputs](https://docs.vllm.ai/en/latest/features/multimodal_inputs.html)
- Companion: [43 VLM Optimization](./VLM_Optimization_Techniques.ipynb) · foundations: [22](./Serving_Fundamentals_KV_Cache_Batching.ipynb), [35](./LongContext_KV_Compression_Serving.ipynb), [39](./Anatomy_Of_A_Decode_Step.ipynb)

▶ **Next:** [VLM Optimization Techniques](./VLM_Optimization_Techniques.ipynb)